In [2]:
!pip install oracledb

   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ---------------------------------- ----- 1.6/1.8 MB 20.8 MB/s eta 0:00:01
   ---------------------------------------- 1.8/1.8 MB 20.2 MB/s  0:00:00
   ---------------------------------------- 0.0/3.5 MB ? eta -:--:--
   ---------------------------------------  3.4/3.5 MB 101.5 MB/s eta 0:00:01
   ---------------------------------------  3.4/3.5 MB 101.5 MB/s eta 0:00:01
   ---------------------------------------  3.4/3.5 MB 101.5 MB/s eta 0:00:01
   ---------------------------------------- 3.5/3.5 MB 4.9 MB/s  0:00:00

   ---------------------------------------- 0/2 [cryptography]
   -------------------- ------------------- 1/2 [oracledb]
   ---------------------------------------- 2/2 [oracledb]



In [3]:
import oracledb

conn = oracledb.connect(
    user="admin",
    password="1234",
    dsn="192.168.0.20:1521/XE"
)

print("DB 연결 성공!")

conn.close()

DB 연결 성공!


In [4]:
import json
import random
from collections import defaultdict

import oracledb


def connect(db_user, db_password, db_host, db_port, db_service_name):
  dsn = f"{db_host}:{db_port}/{db_service_name}"
  return oracledb.connect(user=db_user, password=db_password, dsn=dsn)


def fetch_skill_name_map(conn):
  sql = """
    SELECT SKILL_ID, SKILL_NAME
    FROM SKILL
  """
  cur = conn.cursor()
  cur.execute(sql)
  m = {}
  for skill_id, skill_name in cur:
    m[int(skill_id)] = skill_name
  return m


def fetch_user_skills(conn):
  sql = """
    SELECT USER_ID, SKILL_ID, USER_LEVEL
    FROM USER_SKILL
  """
  cur = conn.cursor()
  cur.execute(sql)

  user_skills = defaultdict(list)
  for user_id, skill_id, level in cur:
    user_skills[user_id].append((int(skill_id), int(level)))
  return user_skills


def fetch_job_postings(conn):
  sql = """
    SELECT POSTING_ID, JOB_CATEGORY
    FROM JOB_POSTING
  """
  cur = conn.cursor()
  cur.execute(sql)

  postings = {}
  for posting_id, job_category in cur:
    postings[str(posting_id).strip()] = {"job_category": job_category}
  return postings


def fetch_job_posting_skills(conn):
  sql = """
    SELECT POSTING_ID, SKILL_ID
    FROM JOB_POSTING_SKILL
  """
  cur = conn.cursor()
  cur.execute(sql)

  job_skills = defaultdict(set)
  for posting_id, skill_id in cur:
    job_skills[str(posting_id).strip()].add(int(skill_id))
  return job_skills


def fetch_category_skill_weight(conn):
  sql = """
    SELECT JOB_CATEGORY, SKILL_ID, WEIGHT
    FROM JOB_CATEGORY_SKILL_WEIGHT
  """
  cur = conn.cursor()
  cur.execute(sql)

  w = {}
  for job_category, skill_id, weight in cur:
    w[(job_category, int(skill_id))] = float(weight)
  return w


def build_user_text(user_skill_rows, skill_name_map):
  names = []
  for skill_id, _level in user_skill_rows:
    name = skill_name_map.get(skill_id)
    if name:
      names.append(name)
  names = sorted(set(names))
  return ", ".join(names)


def build_job_text(posting_id, job_skills, skill_name_map):
  ids = job_skills.get(posting_id, set())
  names = []
  for sid in ids:
    name = skill_name_map.get(sid)
    if name:
      names.append(name)
  names = sorted(set(names))
  return ", ".join(names)


def user_level_weight(level):
  lvl = max(1, min(int(level), 5))
  return lvl / 5.0


def compute_label(user_skill_rows, posting_id, postings, job_skills, cat_skill_weight, alpha=3.0):
  job_cat = postings[posting_id]["job_category"]
  job_set = job_skills.get(posting_id, set())

  total = 0.0
  hit = 0.0

  for sid, lvl in user_skill_rows:
    uw = user_level_weight(lvl)
    total += uw * (1.0 + alpha)  # 최대치 기준으로 정규화

    if sid in job_set:
      cw = cat_skill_weight.get((job_cat, sid), 0.0)
      hit += uw * (1.0 + alpha * cw)

  if total <= 0:
    return 0.0
  score = hit / total

  # 라벨 스케일 확장
  score = score * 4

  if score < 0:
    score = 0.0
  if score > 1:
    score = 1.0

  return score


def pick_candidates_for_user(conn, user_id, pos_k=25, neg_k=75):
  cur = conn.cursor()

  sql_pos = """
    SELECT posting_id
    FROM (
      SELECT jps.POSTING_ID AS posting_id,
             COUNT(*) AS overlap_cnt
      FROM USER_SKILL us
      JOIN JOB_POSTING_SKILL jps
        ON us.SKILL_ID = jps.SKILL_ID
      WHERE us.USER_ID = :user_id
      GROUP BY jps.POSTING_ID
      ORDER BY overlap_cnt DESC
    )
    WHERE ROWNUM <= :pos_k
  """
  cur.execute(sql_pos, {"user_id": user_id, "pos_k": pos_k})
  positives = [str(pid).strip() for (pid,) in cur.fetchall()]

  sql_neg = """
    SELECT posting_id
    FROM (
      SELECT jp.POSTING_ID AS posting_id
      FROM JOB_POSTING jp
      WHERE NOT EXISTS (
        SELECT 1
        FROM USER_SKILL us
        JOIN JOB_POSTING_SKILL jps
          ON us.SKILL_ID = jps.SKILL_ID
        WHERE us.USER_ID = :user_id
          AND jps.POSTING_ID = jp.POSTING_ID
      )
      ORDER BY DBMS_RANDOM.VALUE
    )
    WHERE ROWNUM <= :neg_k
  """
  cur.execute(sql_neg, {"user_id": user_id, "neg_k": neg_k})
  negatives = [str(pid).strip() for (pid,) in cur.fetchall()]

  return positives, negatives


def generate_train_json(
  conn,
  out_path="train.json",
  user_prefix="test",
  per_user_pos=25,
  per_user_neg=75,
  min_job_text_len=1,
  seed=42
):
  random.seed(seed)

  skill_name_map = fetch_skill_name_map(conn)
  user_skills = fetch_user_skills(conn)
  postings = fetch_job_postings(conn)
  job_skills = fetch_job_posting_skills(conn)
  cat_skill_weight = fetch_category_skill_weight(conn)

  # ✅ 디버그(너처럼 0줄 나올 때 원인 잡기)
  print("sample postings keys:", list(postings.keys())[:5])
  print("sample job_skills keys:", list(job_skills.keys())[:5])
  print("job_skills size:", len(job_skills))

  user_ids = [uid for uid in user_skills.keys() if uid.lower().startswith(user_prefix.lower())]
  user_ids.sort()

  total_rows = 0
  wrote_any = False

  # JSON 배열을 스트리밍 방식으로 작성 (메모리 안전)
  with open(out_path, "w", encoding="utf-8") as f:
    f.write('{\n  "data": [\n')

    for user_id in user_ids:
      urows = user_skills.get(user_id, [])
      if not urows:
        continue

      user_text = build_user_text(urows, skill_name_map)
      if not user_text:
        continue

      positives, negatives = pick_candidates_for_user(
        conn,
        user_id,
        pos_k=per_user_pos,
        neg_k=per_user_neg
      )

      for pid in positives + negatives:
        pid = str(pid).strip()

        # postings는 있는데 job_skills가 비면 job_text가 비어서 다 continue 될 수 있음
        if pid not in postings:
          continue

        job_text = build_job_text(pid, job_skills, skill_name_map)
        if len(job_text.strip()) < min_job_text_len:
          continue

        label = compute_label(urows, pid, postings, job_skills, cat_skill_weight)
        rec = {
          "user_id": user_id,
          "posting_id": pid,
          "text1": user_text,
          "text2": job_text,
          "label": round(label, 4)
        }

        if wrote_any:
          f.write(",\n")
        f.write("    " + json.dumps(rec, ensure_ascii=False))
        wrote_any = True
        total_rows += 1

    f.write("\n  ]\n}\n")

  print(f"Done. Wrote {total_rows} rows to {out_path}")


if __name__ == "__main__":
  DB_USER = "admin"
  DB_PASSWORD = "1234"
  DB_HOST = "192.168.0.20"
  DB_PORT = 1521
  DB_SERVICE_NAME = "XE"

  conn = connect(DB_USER, DB_PASSWORD, DB_HOST, DB_PORT, DB_SERVICE_NAME)
  try:
    generate_train_json(
      conn,
      out_path="train.json",
      user_prefix="test",
      per_user_pos=25,
      per_user_neg=75
    )
  finally:
    conn.close()

sample postings keys: ['53045381', '52943594', '53129238', '53088426', '53162023']
sample job_skills keys: ['52969406', '52969496', '52969532', '52969586', '52969669']
job_skills size: 871
Done. Wrote 29921 rows to train.json


In [16]:
import json

with open("train.json", "r", encoding="utf-8") as f:
  data = json.load(f)["data"]

labels = [x["label"] for x in data]
labels.sort()

def pct(p):
  return labels[int(len(labels)*p)]

print("rows:", len(labels))
print("min:", labels[0], "max:", labels[-1])
print("p50:", pct(0.5), "p90:", pct(0.9), "p99:", pct(0.99))

rows: 16102
min: 0.0 max: 0.9785
p50: 0.0 p90: 0.4492 p99: 0.7011


In [1]:
import json
import random
from collections import defaultdict

import oracledb


def connect(db_user, db_password, db_host, db_port, db_service_name):
  dsn = f"{db_host}:{db_port}/{db_service_name}"
  return oracledb.connect(user=db_user, password=db_password, dsn=dsn)


def fetch_skill_name_map(conn):
  cur = conn.cursor()
  cur.execute("SELECT SKILL_ID, SKILL_NAME FROM SKILL")
  return {int(sid): name for sid, name in cur.fetchall()}


def fetch_user_skills(conn, user_prefix):
  cur = conn.cursor()
  cur.execute(
    """
    SELECT USER_ID, SKILL_ID, USER_LEVEL
    FROM USER_SKILL
    WHERE USER_ID LIKE :p
    """,
    {"p": f"{user_prefix}%"},
  )
  m = defaultdict(list)
  for uid, sid, lvl in cur:
    m[uid].append((int(sid), int(lvl)))
  return m


def fetch_job_postings(conn):
  cur = conn.cursor()
  cur.execute("SELECT POSTING_ID, JOB_CATEGORY FROM JOB_POSTING")
  return {str(pid).strip(): {"job_category": cat} for pid, cat in cur.fetchall()}


def fetch_job_posting_skills(conn):
  cur = conn.cursor()
  cur.execute("SELECT POSTING_ID, SKILL_ID FROM JOB_POSTING_SKILL")
  m = defaultdict(set)
  for pid, sid in cur:
    m[str(pid).strip()].add(int(sid))
  return m


def fetch_category_skill_weight(conn):
  cur = conn.cursor()
  cur.execute("SELECT JOB_CATEGORY, SKILL_ID, WEIGHT FROM JOB_CATEGORY_SKILL_WEIGHT")
  return {(cat, int(sid)): float(w) for cat, sid, w in cur.fetchall()}


def user_level_weight(level):
  lvl = max(1, min(int(level), 5))
  return lvl / 5.0


def build_user_text(user_skill_rows, skill_name_map):
  names = sorted({skill_name_map.get(sid) for sid, _lvl in user_skill_rows if sid in skill_name_map})
  return ", ".join([n for n in names if n])


def build_job_text(posting_id, job_skills, skill_name_map):
  ids = job_skills.get(posting_id, set())
  names = sorted({skill_name_map.get(sid) for sid in ids if sid in skill_name_map})
  return ", ".join([n for n in names if n])


def compute_label(user_skill_rows, posting_id, postings, job_skills, cat_skill_weight, alpha=3.0, scale=4.0):
  job_cat = postings[posting_id]["job_category"]
  job_set = job_skills.get(posting_id, set())

  total = 0.0
  hit = 0.0

  for sid, lvl in user_skill_rows:
    uw = user_level_weight(lvl)
    total += uw * (1.0 + alpha)
    if sid in job_set:
      cw = cat_skill_weight.get((job_cat, sid), 0.0)
      hit += uw * (1.0 + alpha * cw)

  if total <= 0:
    return 0.0

  score = (hit / total) * scale
  return max(0.0, min(1.0, score))


def pick_test_candidates_for_user(conn, user_id, pos_k=10, neg_k=30):
  """
  pos: 유저 스킬과 겹치는 공고 중 overlap 많은 순 상위 pos_k
  neg: 유저 스킬과 겹치는 공고가 '없는' 공고 중 랜덤 neg_k
  """
  cur = conn.cursor()

  sql_pos = """
    SELECT posting_id
    FROM (
      SELECT jp.POSTING_ID AS posting_id,
             COUNT(*) AS overlap_cnt
      FROM USER_SKILL us
      JOIN JOB_POSTING_SKILL jps ON us.SKILL_ID = jps.SKILL_ID
      JOIN JOB_POSTING jp ON jp.POSTING_ID = jps.POSTING_ID
      WHERE us.USER_ID = :user_id
      GROUP BY jp.POSTING_ID
      ORDER BY overlap_cnt DESC
    )
    WHERE ROWNUM <= :k
  """
  cur.execute(sql_pos, {"user_id": user_id, "k": pos_k})
  positives = [str(pid).strip() for (pid,) in cur.fetchall()]

  sql_neg = """
    SELECT posting_id
    FROM (
      SELECT jp.POSTING_ID AS posting_id
      FROM JOB_POSTING jp
      WHERE NOT EXISTS (
        SELECT 1
        FROM USER_SKILL us
        JOIN JOB_POSTING_SKILL jps ON us.SKILL_ID = jps.SKILL_ID
        WHERE us.USER_ID = :user_id
          AND jps.POSTING_ID = jp.POSTING_ID
      )
      ORDER BY DBMS_RANDOM.VALUE
    )
    WHERE ROWNUM <= :k
  """
  cur.execute(sql_neg, {"user_id": user_id, "k": neg_k})
  negatives = [str(pid).strip() for (pid,) in cur.fetchall()]

  return positives, negatives


def generate_test_json(
  conn,
  out_path="test.json",
  user_prefix="test_make_user",
  per_user_pos=10,
  per_user_neg=30,
  alpha=3.0,
  scale=4.0,
  seed=42
):
  random.seed(seed)

  skill_name_map = fetch_skill_name_map(conn)
  user_skills = fetch_user_skills(conn, user_prefix=user_prefix)
  postings = fetch_job_postings(conn)
  job_skills = fetch_job_posting_skills(conn)
  cat_skill_weight = fetch_category_skill_weight(conn)

  user_ids = sorted(user_skills.keys())

  total_rows = 0
  wrote_any = False

  with open(out_path, "w", encoding="utf-8") as f:
    f.write('{\n  "data": [\n')

    for user_id in user_ids:
      urows = user_skills.get(user_id, [])
      if not urows:
        continue

      user_text = build_user_text(urows, skill_name_map)
      if not user_text.strip():
        continue

      positives, negatives = pick_test_candidates_for_user(
        conn,
        user_id,
        pos_k=per_user_pos,
        neg_k=per_user_neg
      )

      for pid in positives + negatives:
        pid = str(pid).strip()
        if pid not in postings:
          continue

        job_text = build_job_text(pid, job_skills, skill_name_map)
        if not job_text.strip():
          continue

        label = compute_label(urows, pid, postings, job_skills, cat_skill_weight, alpha=alpha, scale=scale)
        rec = {
          "user_id": user_id,
          "posting_id": pid,
          "text1": user_text,
          "text2": job_text,
          "label": round(label, 4)
        }

        if wrote_any:
          f.write(",\n")
        f.write("    " + json.dumps(rec, ensure_ascii=False))
        wrote_any = True
        total_rows += 1

    f.write("\n  ]\n}\n")

  print(f"Done. Wrote {total_rows} rows to {out_path}")


if __name__ == "__main__":
  DB_USER = "admin"
  DB_PASSWORD = "1234"
  DB_HOST = "192.168.0.20"
  DB_PORT = 1521
  DB_SERVICE_NAME = "XE"

  conn = connect(DB_USER, DB_PASSWORD, DB_HOST, DB_PORT, DB_SERVICE_NAME)
  try:
    generate_test_json(
      conn,
      out_path="test.json",
      user_prefix="test_make_user",
      per_user_pos=10,
      per_user_neg=30,
      alpha=3.0,
      scale=4.0
    )
  finally:
    conn.close()

Done. Wrote 5890 rows to test.json


In [3]:
!pip install sentence_transformers

  Using cached sentence_transformers-5.1.2-py3-none-any.whl.metadata (16 kB)
  Using cached torch-2.8.0-cp39-cp39-win_amd64.whl.metadata (30 kB)
  Using cached filelock-3.19.1-py3-none-any.whl.metadata (2.1 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-win_amd64.whl.metadata (7.4 kB)
  Using cached safetensors-0.7.0-cp38-abi3-win_amd64.whl.metadata (4.2 kB)
  Using cached fsspec-2025.10.0-py3-none-any.whl.metadata (10 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.2.1-py3-none-any.whl.metadata (5.2 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
Using cached sentence_transformers-5.1.2-py3-none-any.whl (488 kB)
   ---------------------------------------- 0.0/12.0 MB ? eta -:--:--
   ---------------------------------------- 12.0/12.0 MB 68.2 MB/s  0:00:00
   ---------------------------------------- 0.0/566.4 kB ? eta -:--:--
   ---------------------------------------- 566.4/566.4 kB ?  0:00:00
Using cached tokenizers-0.22.

In [4]:
import json
import numpy as np
from collections import defaultdict
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity


model = SentenceTransformer("sbert_specup_model")


with open("test.json","r",encoding="utf-8") as f:
    data = json.load(f)["data"]


user_groups = defaultdict(list)

for row in data:
    user_groups[row["user_id"]].append(row)


def dcg(scores):
    return sum(
        (2**s - 1) / np.log2(i + 2)
        for i, s in enumerate(scores)
    )


def ndcg(scores, k):

    scores = scores[:k]
    best = sorted(scores, reverse=True)

    return dcg(scores) / (dcg(best) + 1e-9)


hit5 = []
hit10 = []

ndcg5 = []
ndcg10 = []


for user, rows in user_groups.items():

    texts1 = [r["text1"] for r in rows]
    texts2 = [r["text2"] for r in rows]
    labels = [r["label"] for r in rows]

    emb1 = model.encode(texts1)
    emb2 = model.encode(texts2)

    sims = cosine_similarity(emb1, emb2).diagonal()

    idx = np.argsort(-sims)

    ranked_labels = [labels[i] for i in idx]

    # HitRate
    hit5.append(max(ranked_labels[:5]) > 0)
    hit10.append(max(ranked_labels[:10]) > 0)

    # NDCG
    ndcg5.append(ndcg(ranked_labels,5))
    ndcg10.append(ndcg(ranked_labels,10))


print("HitRate@5 :", np.mean(hit5))
print("HitRate@10:", np.mean(hit10))

print("NDCG@5 :", np.mean(ndcg5))
print("NDCG@10:", np.mean(ndcg10))

c:\Users\KOSMO\.conda\envs\MyCustomEnv39\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
No sentence-transformers model found with name sentence-transformers/sbert_specup_model. Creating a new one with mean pooling.


OSError: sentence-transformers/sbert_specup_model is not a local folder and is not a valid model identifier listed on 'https://huggingface.co/models'
If this is a private repository, make sure to pass a token having permission to this repo either by logging in with `hf auth login` or by passing `token=<your_token>`